In [9]:

import os
import platform
import time
import statistics

import psutil
import torch
import torch.nn as nn

from torchvision.models import mobilenet_v2


In [10]:

print("========== SYSTEM ==========")
print("OS:", platform.platform())
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)

print("\n========== CPU ==========")
print("Processor:", platform.processor())
print("Physical cores:", psutil.cpu_count(logical=False))
print("Logical cores:", psutil.cpu_count(logical=True))
print("CPU frequency:", psutil.cpu_freq())

print("\n========== RAM ==========")
print("Total:",
      round(psutil.virtual_memory().total / 1024**3, 2), "GB")
print("Available:",
      round(psutil.virtual_memory().available / 1024**3, 2), "GB")

print("\n========== GPU ==========")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("No CUDA GPU available.")


# Select the available device
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Keep CPU thread configuration consistent
CPU_THREADS = min(2, os.cpu_count() or 1)
torch.set_num_threads(CPU_THREADS)

# Benchmark configuration
BATCH_SIZE = 32
NUM_CLASSES = 10
IMAGE_SIZE = 64

print("\n========== CONFIGURATION ==========")
print("Device:", DEVICE)
print("CPU threads:", torch.get_num_threads())
print("Batch size:", BATCH_SIZE)
print("Input shape:", (BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE))
print("Available RAM:",
      round(psutil.virtual_memory().available / 1024**3, 2), "GB")


========== SYSTEM ==========
OS: Windows-10-10.0.19045-SP0
Python: 3.12.4
PyTorch: 2.10.0+cpu

========== CPU ==========
Processor: Intel64 Family 6 Model 142 Stepping 9, GenuineIntel
Physical cores: 2
Logical cores: 4
CPU frequency: scpufreq(current=2701.0, min=0.0, max=2701.0)

========== RAM ==========
Total: 11.87 GB
Available: 2.19 GB

========== GPU ==========
CUDA available: False
No CUDA GPU available.

========== CONFIGURATION ==========
Device: cpu
CPU threads: 2
Batch size: 32
Input shape: (32, 3, 64, 64)
Available RAM: 2.19 GB


In [5]:

# Reproducibility
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Create model
model = mobilenet_v2(
    weights=None,
    num_classes=NUM_CLASSES
).to(DEVICE)

# Generate one synthetic batch
# Values are in [-1, 1], matching the normalization in task.py
images = torch.rand(
    BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE
) * 2 - 1

images = images.to(DEVICE)

labels = torch.randint(
    0, NUM_CLASSES, (BATCH_SIZE,)
).to(DEVICE)

# Loss function
criterion = nn.CrossEntropyLoss().to(DEVICE)

# Optimizer matching task.py
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9
)

# Number of trainable parameters
num_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Model: MobileNetV2")
print("Trainable parameters:", f"{num_parameters:,}")
print("Model created successfully.")


Model: MobileNetV2
Trainable parameters: 2,236,682
Model created successfully.


In [11]:

def synchronize_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)


def training_step():
    model.train()

    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    return loss.item()


# Warm-up
WARMUP_STEPS = 20

for _ in range(WARMUP_STEPS):
    training_step()

synchronize_device()


# Benchmark configuration
NUM_TRIALS = 9
STEPS_PER_TRIAL = 10

trial_times = []

for trial in range(NUM_TRIALS):

    synchronize_device()
    start = time.perf_counter()

    for _ in range(STEPS_PER_TRIAL):
        training_step()

    synchronize_device()
    elapsed = time.perf_counter() - start

    trial_times.append(elapsed)


# Calculate statistics
median_time = statistics.median(trial_times)

quartiles = statistics.quantiles(
    trial_times,
    n=4,
    method="inclusive"
)

q1 = quartiles[0]
q3 = quartiles[2]
iqr = q3 - q1

time_per_batch = median_time / STEPS_PER_TRIAL
throughput = BATCH_SIZE / time_per_batch


# Results
print("========== TRAINING BENCHMARK ==========")
print("Device:", DEVICE)
print("Optimizer: SGD")
print("Momentum: 0.9")
print("Warm-up batches:", WARMUP_STEPS)
print("Trials:", NUM_TRIALS)
print("Batches per trial:", STEPS_PER_TRIAL)

print("\nTrial times:",
      [round(t, 3) for t in trial_times])

print("\nMedian time per 10 batches:",
      round(median_time, 3), "seconds")

print("Q1:", round(q1, 3), "seconds")
print("Q3:", round(q3, 3), "seconds")
print("IQR:", round(iqr, 3), "seconds")

print("\nMedian time per batch:",
      round(time_per_batch, 4), "seconds")

print("Training throughput:",
      round(throughput, 2), "images/second")


========== TRAINING BENCHMARK ==========
Device: cpu
Optimizer: SGD
Momentum: 0.9
Warm-up batches: 20
Trials: 9
Batches per trial: 10

Trial times: [5.917, 6.484, 5.806, 6.173, 6.014, 5.376, 5.432, 5.224, 5.236]

Median time per 10 batches: 5.806 seconds
Q1: 5.376 seconds
Q3: 6.014 seconds
IQR: 0.638 seconds

Median time per batch: 0.5806 seconds
Training throughput: 55.11 images/second


In [12]:

model.eval()

# Warm-up
with torch.no_grad():
    for _ in range(5):
        model(images)

synchronize_device()

# Benchmark
NUM_RUNS = 30
inference_times = []

with torch.no_grad():
    for _ in range(NUM_RUNS):

        synchronize_device()
        start = time.perf_counter()

        model(images)

        synchronize_device()
        elapsed = time.perf_counter() - start

        inference_times.append(elapsed)


median_inference_time = statistics.median(inference_times)

print("========== INFERENCE BENCHMARK ==========")
print("Median batch inference time:",
      round(median_inference_time * 1000, 3), "ms")

print("Median inference time per image:",
      round(median_inference_time / BATCH_SIZE * 1000, 3), "ms")


========== INFERENCE BENCHMARK ==========
Median batch inference time: 104.289 ms
Median inference time per image: 3.259 ms


In [13]:

process = psutil.Process(os.getpid())

# Initial process memory
memory_before = process.memory_info().rss / (1024 ** 2)

peak_memory = memory_before

# Reset GPU peak memory counter
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats(DEVICE)

# Run a separate training workload
MEMORY_STEPS = 30

for _ in range(MEMORY_STEPS):

    training_step()

    current_memory = (
        process.memory_info().rss / (1024 ** 2)
    )

    peak_memory = max(peak_memory, current_memory)

synchronize_device()

memory_after = process.memory_info().rss / (1024 ** 2)

print("========== MEMORY BENCHMARK ==========")
print("Memory before:", round(memory_before, 2), "MB")
print("Memory after:", round(memory_after, 2), "MB")

print("Peak sampled process memory:",
      round(peak_memory, 2), "MB")

print("Peak increase over initial:",
      round(peak_memory - memory_before, 2), "MB")

print("Available system RAM:",
      round(psutil.virtual_memory().available / 1024**3, 2),
      "GB")

if DEVICE.type == "cuda":
    gpu_peak = torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 2)
    print("Peak GPU allocated memory:", round(gpu_peak, 2), "MB")


========== MEMORY BENCHMARK ==========
Memory before: 468.78 MB
Memory after: 509.93 MB
Peak sampled process memory: 513.13 MB
Peak increase over initial: 44.35 MB
Available system RAM: 2.63 GB
